# 10. The Full Manual RAG Pipeline: Search + Prompt + Agent

**⚠️ Dependency:** this script requires `"cloudxeus-support-rag-agent"` to already exist — the agent created in
`09_customer_rag_agent.py`. Run that script first.

This is the complete "manual RAG" pipeline this section has been building toward across `08_ai_search.py` and
`09_customer_rag_agent.py`: the `ask(question)` function (1) queries Azure AI Search with a **hybrid** query
(vector + keyword) to retrieve relevant document chunks, (2) formats those chunks into a "Sources:" block, (3)
builds a full grounded prompt combining instructions, sources, and the customer's question, and (4) sends that
constructed prompt as `input` to the `cloudxeus-support-rag-agent` via the Responses API. Unlike
`08_ai_search.py`'s plain keyword search, this uses `VectorizableTextQuery` for **integrated vectorization** —
the search service embeds the query text itself, so the client never has to call an embedding model directly.

**Difficulty:** Advanced


## Prerequisites

**pip3 packages:**
- `azure-search-documents` — not in root `requirements.txt`, install with `pip3 install azure-search-documents`
- `azure-ai-projects`, `azure-identity`, `python-dotenv` — already in root `requirements.txt`

**Azure resource(s) required:**
- The Foundry project with `cloudxeus-support-rag-agent` already created (`09_customer_rag_agent.py`).
- An Azure AI Search index (same one queried in `08_ai_search.py`) that has **integrated vectorization**
  configured — i.e. a vectorizer/embedding skill attached to the `text_vector` field, since `VectorizableTextQuery`
  relies on the search service to embed the query text server-side.

**Environment variables** (add to the root `.env`):
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_SEARCH_ENDPOINT`
- `AZURE_SEARCH_INDEX_NAME`
- `AZURE_AI_RAG_AGENT_NAME` (optional, defaults to `"cloudxeus-support-rag-agent"`)


## What You'll Learn

- How `VectorizableTextQuery` performs **integrated vectorization** — the search service embeds your query
  text using its configured vectorizer, so you never call an embedding API yourself
- How combining `search_text=` (keyword) with `vector_queries=[...]` (vector) in one `.search()` call performs
  **hybrid search**
- How to format retrieved chunks into a clearly delimited "Sources:" block for prompt injection
- How to combine retrieval + prompt construction + agent invocation into a single reusable `ask()` function


### Setup: project client, OpenAI client, and search client

Three clients are constructed up front: `AIProjectClient` (for Foundry project access),
`project.get_openai_client()` (the Responses-API-shaped client used to invoke the agent, same as
`03_invoke_agent.py`/`07_helpdesk_client.py`), and a `SearchClient` (same shape as `08_ai_search.py`, now
pointed at the environment-driven `SEARCH_ENDPOINT`/`INDEX_NAME`).

- **💡 Exam tip:** Notice this script talks to **two separate Azure resources** — Azure AI Foundry (for the
  agent) and Azure AI Search (for retrieval) — each with its own endpoint and its own RBAC role requirements
  on your identity. AI-102 scenarios often involve exactly this kind of multi-resource architecture.
- **🔄 Alternatives:** Rather than maintaining two separate clients and stitching them together yourself, you
  could register the Azure AI Search index directly as a knowledge source on the agent (e.g. via
  `AzureAISearchTool`), letting Foundry perform retrieval server-side — see the "Try It Yourself" section for
  more on this trade-off.


In [ ]:
import os

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-1782198581571")
AGENT_NAME = os.getenv("AZURE_AI_RAG_AGENT_NAME", "cloudxeus-support-rag-agent")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

openai_client = project.get_openai_client()

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=DefaultAzureCredential(),
)


### Steps 1–4: hybrid retrieval, source formatting, grounded prompt, and agent invocation

The whole pipeline lives in one function, `ask(question)`, so it stays in a single code cell below (splitting
a function's body across notebook cells doesn't work — each cell must be independently valid Python).
Walking through it:

- **Step 1 — hybrid retrieval:** `search_client.search(...)` is called with **both**
  `search_text=question` (keyword) and `vector_queries=[VectorizableTextQuery(text=question,
  k_nearest_neighbors=3, fields="text_vector")]` (vector) — this is hybrid search: Azure AI Search runs both
  retrieval modes and merges/reranks the combined results, returning the top `top=3` overall.
- **Step 2 — formatting sources:** each result's `chunk`, `title`, and `parent_id` are pulled out and
  formatted into a labeled source block; all blocks are joined with blank lines into one `sources` string.
- **Step 3 — building the grounded prompt:** the `sources` block is embedded into a `prompt` string alongside
  instructions ("answer only from the sources below") and the customer's original `question`.
- **Step 4 — invoking the agent:** the constructed `prompt` is sent as `input` to
  `openai_client.responses.create(...)`, routed to `cloudxeus-support-rag-agent` via `agent_reference` — note
  this call has **no `version`** specified, so it uses that agent's latest version, the same ambiguity as
  `03_invoke_agent.py`. The function returns `response.output_text`, the agent's final grounded answer.

- **💡 Exam tip:** `k_nearest_neighbors=3` controls how many nearest-neighbor vector matches to consider for
  the *vector* side of the query specifically; `top=3` caps the final combined result count returned overall.
  Azure AI Search merges vector and keyword result sets (commonly via Reciprocal Rank Fusion) when both are
  present in one query — recognize "hybrid search" as its own retrieval mode, distinct from either alone. Also
  note this is "prompt-stuffing RAG" in its most explicit form: retrieval, grounding instructions, and the
  question are all concatenated into one string the model sees as a single turn — it works with any agent
  regardless of whether that agent has tools, because from the model's point of view it's just answering a
  detailed question.
- **🔄 Alternatives:** You could compute the query embedding yourself (via an embeddings model) and pass a raw
  vector instead of `VectorizableTextQuery` — that requires managing an embedding client and ensuring the
  embedding model matches what the index was built with, which integrated vectorization avoids entirely. Also,
  keeping the "answer only from sources" instruction in the **agent's own system prompt** (as
  `09_customer_rag_agent.py` does) rather than repeating it in every `ask()` call's `prompt` here is slightly
  redundant — you could simplify `prompt` to just the sources + question, relying on the agent's
  already-configured instructions for the grounding rules.


In [ ]:
def ask(question):
    # Step 1: Retrieve grounding data from Azure AI Search (hybrid: keyword + vector)
    results = search_client.search(
        search_text=question,
        vector_queries=[
            VectorizableTextQuery(
                text=question,
                k_nearest_neighbors=3,
                fields="text_vector",
            )
        ],
        select=["chunk", "title", "parent_id"],
        top=3,
    )

    # Step 2: Convert search results into source text
    source_list = []

    for result in results:
        title = result.get("title", "Unknown source")
        parent_id = result.get("parent_id", "")
        chunk = result.get("chunk", "")

        source_text = f"""
[Source title: {title}]
[Parent ID: {parent_id}]
{chunk}
"""
        source_list.append(source_text)

    sources = "\n\n".join(source_list)

    # Step 3: Build the grounded prompt for the agent
    prompt = f"""
You are a customer support agent for CloudXeus Technology Services.

Answer the customer question using only the sources provided below.
If the sources do not contain enough information, say that the knowledge base does not contain enough information.

Sources:
{sources}

Customer question:
{question}
"""

    print("\nPrompt sent to agent:\n")
    print(prompt)

    # Step 4: Ask the Foundry agent to answer using the sources
    response = openai_client.responses.create(
        extra_body={
            "agent_reference": {
                "name": AGENT_NAME,
                "type": "agent_reference"
            }
        },
        input=prompt,
    )

    return response.output_text


print(ask("Can I get my money back?"))


## Summary

`ask("Can I get my money back?")` runs the full manual RAG pipeline: hybrid search retrieves the top 3
relevant chunks from Azure AI Search, those chunks are formatted into a sources block, a grounded prompt is
assembled from instructions + sources + the question, and the Foundry `cloudxeus-support-rag-agent` produces
a final answer — which should cite the source it used and refuse to answer beyond what the retrieved chunks
support, per the agent's system prompt from `09_customer_rag_agent.py`.


## Try It Yourself

1. **Easy:** Call `ask(...)` with a question you know is **not** covered by the indexed documents and confirm
   the agent responds with the "I don't have that information..." fallback rather than guessing.
2. **Intermediate:** Change `k_nearest_neighbors` and `top` to larger values (e.g. 5) and compare how much
   longer/more detailed the constructed prompt and final answer become.
3. **Advanced:** Refactor `ask()` to skip the manual prompt-construction step and instead attach an
   `AzureAISearchTool` (pointed at the same index) directly to a *new* agent definition, letting Foundry
   perform retrieval automatically — compare the resulting code's complexity and the two approaches' answers
   for the same question.
